In [1]:
%%info

In [2]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "/usr/bin/python3",
        "spark.pyspark.driver.python": "/usr/bin/python3"
    }
}

In [3]:
import pandas as pd
import numpy as np
import io
import os
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1781034493459_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
PATH = 's3://p11-s2-bucket-197446685131-eu-north-1-an'
PATH_Data = PATH+'/Test'
PATH_Result = PATH+'/Results'
print('PATH:        '+\
	PATH+'\nPATH_Data:   '+\
	PATH_Data+'\nPATH_Result: '+PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://p11-s2-bucket-197446685131-eu-north-1-an
PATH_Data:   s3://p11-s2-bucket-197446685131-eu-north-1-an/Test
PATH_Result: s3://p11-s2-bucket-197446685131-eu-north-1-an/Results

In [5]:
images = spark.read.format("binaryFile") \
	.option("pathGlobFilter", "*.jpg") \
	.option("recursiveFileLookup", "true") \
	.load(PATH_Data)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
images.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://p11-s2-bucke...|2026-06-08 12:14:56|  7353|[FF D8 FF E0 00 1...|
|s3://p11-s2-bucke...|2026-06-08 12:14:56|  7350|[FF D8 FF E0 00 1...|
|s3://p11-s2-bucke...|2026-06-08 12:14:56|  7349|[FF D8 FF E0 00 1...|
|s3://p11-s2-bucke...|2026-06-08 12:14:56|  7348|[FF D8 FF E0 00 1...|
|s3://p11-s2-bucke...|2026-06-08 12:14:57|  7328|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

In [7]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+---------------------------------------------------------------------------+----------+
|path                                                                       |label     |
+---------------------------------------------------------------------------+----------+
|s3://p11-s2-bucket-197446685131-eu-north-1-an/Test/Watermelon/r_106_100.jpg|Watermelon|
|s3://p11-s2-bucket-197446685131-eu-north-1-an/Test/Watermelon/r_109_100.jpg|Watermelon|
|s3://p11-s2-bucket-197446685131-eu-north-1-an/Test/Watermelon/r_108_100.jpg|Watermelon|
|s3://p11-s2-bucket-197446685131-eu-north-1-an/Test/Watermelon/r_107_100.jpg|Watermelon|
|s3://p11-s2-bucket-197446685131-eu-north-1-an/Test/Watermelon/r_95_100.jpg |Watermelon|
+---------------------------------------------------------------------------+-

In [8]:
model = MobileNetV2(weights='imagenet',
					include_top=True,
					input_shape=(224, 224, 3))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

In [9]:
new_model = Model(inputs=model.input,
				  outputs=model.layers[-2].output)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
brodcast_weights = sc.broadcast(new_model.get_weights())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
new_model.summary()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │

In [12]:
def model_fn():
	"""
	Returns a MobileNetV2 model with top layer removed 
	and broadcasted pretrained weights.
	"""
	model = MobileNetV2(weights=None,
						include_top=True,
						input_shape=(224, 224, 3))
	for layer in model.layers:
		layer.trainable = False
	new_model = Model(inputs=model.input,
				  outputs=model.layers[-2].output)
	new_model.set_weights(brodcast_weights.value)
	return new_model

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
def preprocess(content):
	"""
	Preprocesses raw image bytes for prediction.
	"""
	img = Image.open(io.BytesIO(content)).resize([224, 224])
	arr = img_to_array(img)
	return preprocess_input(arr)

def featurize_series(model, content_series):
	"""
	Featurize a pd.Series of raw images using the input model.
	:return: a pd.Series of image features
	"""
	input = np.stack(content_series.map(preprocess))
	preds = model.predict(input)
	# For some layers, output features will be multi-dimensional tensors.
	# We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
	output = [p.flatten() for p in preds]
	return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
	'''
	This method is a Scalar Iterator pandas UDF wrapping our featurization function.
	The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

	:param content_series_iter: This argument is an iterator over batches of data, where each batch
							  is a pandas Series of image data.
	'''
	# With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
	# for multiple data batches.  This amortizes the overhead of loading big models.

	os.environ['OMP_NUM_THREADS'] = '1'
	os.environ['MKL_NUM_THREADS'] = '1'
	os.environ['TF_NUM_INTRA_OP_PARALLELISM_THREADS'] = '1'
	os.environ['TF_NUM_INTER_OP_PARALLELISM_THREADS'] = '1'

	import tensorflow as tf
	tf.config.threading.set_intra_op_parallelism_threads(1)
	tf.config.threading.set_inter_op_parallelism_threads(1)

	model = model_fn()
	for content_series in content_series_iter:
		yield featurize_series(model, content_series)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/mnt1/yarn/usercache/livy/appcache/application_1781034493459_0002/container_1781034493459_0002_01_000001/pyspark.zip/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

In [14]:
features_df = images.repartition(24).select(col("path"),
											col("label"),
											featurize_udf("content").alias("features")
										   )

features_df.cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[path: string, label: string, features: array<float>]

In [15]:
from pyspark.ml.feature import PCA
from pyspark.ml.functions import array_to_vector
from pyspark.ml.functions import vector_to_array

features_df = features_df.withColumn("features_vec", array_to_vector("features"))

pca = PCA(k=100, inputCol="features_vec", outputCol="reduced_vec")
pca_model = pca.fit(features_df)

features_df = pca_model.transform(features_df)

features_df = features_df.drop("features_vec").drop("features")

features_df = features_df.withColumn("reduced_vec", vector_to_array("reduced_vec"))

features_df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+--------------------+
|                path|         label|         reduced_vec|
+--------------------+--------------+--------------------+
|s3://p11-s2-bucke...|    Watermelon|[-2.6043106879126...|
|s3://p11-s2-bucke...|    Watermelon|[-2.0292285684011...|
|s3://p11-s2-bucke...|    Watermelon|[-2.4219530270332...|
|s3://p11-s2-bucke...|Pineapple Mini|[-4.5107355151103...|
|s3://p11-s2-bucke...|Pineapple Mini|[-5.6058437162277...|
+--------------------+--------------+--------------------+
only showing top 5 rows

In [16]:
print(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://p11-s2-bucket-197446685131-eu-north-1-an/Results

In [17]:
features_df.write.mode("overwrite").parquet(PATH_Result)

features_df.unpersist()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[path: string, label: string, reduced_vec: array<double>]

In [18]:
print("Validation des résultats stockés sur S3 :")
df = spark.read.parquet(PATH_Result)

df.show(5, truncate=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Validation des r?sultats stock?s sur S3 :
+--------------------+--------------+--------------------+
|                path|         label|         reduced_vec|
+--------------------+--------------+--------------------+
|s3://p11-s2-bucke...|    Watermelon|[-4.2047744069245...|
|s3://p11-s2-bucke...|    Watermelon|[-2.1068547135485...|
|s3://p11-s2-bucke...|    Watermelon|[-3.5098665147429...|
|s3://p11-s2-bucke...|    Watermelon|[-1.7406180604206...|
|s3://p11-s2-bucke...|Pineapple Mini|[-6.2244858171086...|
+--------------------+--------------+--------------------+
only showing top 5 rows

In [19]:
df.select("reduced_vec").show(1, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
print(f"Nombre total de lignes traitées : {df.count()}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Nombre total de lignes trait?es : 22688